# Machine Learning Basics

This is a continuation of the prior lecture's notebook, demonstrating the perforance for KRR.


## Install Dependencies


In [ ]:
!uv pip install matminer scikit-learn

## Load the Dataset

We use the `expt_formation_enthalpy_kingsbury` dataset from matminer, which contains experimentally measured formation enthalpies for ~1500 inorganic compounds. For faster computation during this demo, we limit to the first 1000 entries.

The available datasets can be found on the [matminer website](https://hackingmaterials.lbl.gov/matminer/dataset_summary.html).

The dataset is loaded as a `pandas` dataframe object, which is a convenient format for tabular data. The syntax can be a bit clunky the first time you use it, but that is not our focus here today. Also, LLMs know most things about `pandas` if you need some support.


In [ ]:
from matminer.datasets.dataset_retrieval import load_dataset

df = load_dataset("expt_formation_enthalpy_kingsbury")
df = df[0:1000]  # limit for quicker calculations
print(f"Dataset loaded: {df.shape[0]} entries, {df.shape[1]} columns")

Let's take a look at what this dataframe has inside of it.


In [ ]:
df.head(10)

## Featurize the Compositions

Machine learning models require numerical input. We convert each chemical formula into a `Composition` object and then generate some statistical properties of the elements for the features:

- **Statistics**: minimum, maximum, range, mean, standard deviation across constituent elements
- **Properties**: atomic number, atomic weight, periodic table column/row, covalent radius, electronegativity

This yields a fixed-length numerical feature vector for each compound. You can find a list of the supported elemental properties and their values [here](https://github.com/hackingmaterials/matminer/tree/dfc0f4c830fb9eb9c3366bdb26be2722fd09ac06/src/matminer/utils/data_files/magpie_elementdata) and the supported statical properties [here](https://hackingmaterials.lbl.gov/matminer/matminer.featurizers.utils.html#module-matminer.featurizers.utils.stats).


In [ ]:
from pymatgen.core import Composition

# Convert formula strings to Composition objects
df["composition"] = df["formula"].apply(Composition)

In [ ]:
from matminer.featurizers.composition import ElementProperty

# Define which statistics and element properties to use
stats = ["minimum", "maximum", "range", "mean", "std_dev"]
features = [
    "Number",
    "AtomicWeight",
    "Column",
    "Row",
    "CovalentRadius",
    "Electronegativity",
]

# Build and apply the featurizer
featurizer = ElementProperty(data_source="magpie", features=features, stats=stats)
df = featurizer.featurize_dataframe(df, "composition")

print(f"Feature matrix shape: {df.filter(like='Magpie').shape}")

## Data Inspection


Now let's take a look at our updated dataframe.


In [ ]:
df.head(5)

## Prepare Features and Target

We extract:

- `X`: the feature matrix (input)
- `y`: experimental formation enthalpy in eV/atom (target)


In [ ]:
X = df.filter(like="Magpie").values
y = df["expt_form_e"].values

print(f"X shape: {X.shape}  →  {X.shape[0]} samples, {X.shape[1]} features")
print(f"y shape: {y.shape}")

### Three-Way Split

We carve 20 % off for testing, then split the remaining 80 % again into a training and held-out validaiton set. Of course, what you choose to do here is in practice up to you.


In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: hold out the test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Step 2: split the remainder into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.2,
    random_state=42,
)

print(f"Train:      {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test:       {X_test.shape[0]} samples")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit + transform on train
X_val_sc = scaler.transform(X_val)  # transform only on validation
X_test_sc = scaler.transform(X_test)  # transform only on testing

# Kernel Ridge Regression


We can do basically the same process as before but now for KRR. The performance will be greatly improved because we can introduce nonlinearity.


In [ ]:
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_absolute_error
import numpy as np

# Define hyperparameters to sweep over
alphas = np.logspace(-4, 0, 5)  # regularization parameter
gammas = np.logspace(-4, 0, 5)  # kernel function parameter\
kernel = "rbf"
best_val_mae = np.inf

for alpha in alphas:  # regularizaiton strength
    for gamma in gammas:  # for kernel function
        # Train model
        model = KernelRidge(alpha=alpha, gamma=gamma, kernel=kernel)
        model.fit(X_train_sc, y_train)

        # Evaluation on validation set
        val_mae = mean_absolute_error(y_val, model.predict(X_val_sc))

        # Determine if these are the best parameters
        if val_mae < best_val_mae:
            best_alpha = alpha
            best_gamma = gamma
            best_val_mae = val_mae

print(f"α = {best_alpha}, γ = {best_gamma}, Val MAE = {best_val_mae} eV/atom")

Now we evaluate model performance with the optimal hyperparameters...


In [ ]:
# Scale train+val together (refit scaler on larger set)
scaler = StandardScaler()
X_trainval_sc = scaler.fit_transform(X_trainval)
X_test_sc = scaler.transform(X_test)

In [ ]:
# Train model on combined training and validation set
model = KernelRidge(alpha=best_alpha, gamma=best_gamma, kernel=kernel)
model.fit(X_trainval_sc, y_trainval)
y_test_pred = model.predict(X_test_sc)

In [ ]:
mae_test = mean_absolute_error(y_test, y_test_pred)
print(f"Testing MAE = {mae_test} eV/atom")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(y_test, y_test_pred, alpha=0.6, edgecolors="k", linewidths=0.3, s=35)
lims = [
    min(y_test.min(), y_test_pred.min()) - 0.1,
    max(y_test.max(), y_test_pred.max()) + 0.1,
]
ax.plot(lims, lims, "r--", label="y=x")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Actual Formation Enthalpy (eV/atom)")
ax.set_ylabel("Predicted Formation Enthalpy (eV/atom)")
ax.set_title("Kernel Ridge Regression – Test Set")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()